<a href="https://colab.research.google.com/github/Luoxiaowanzi/Class/blob/main/Copy_of_assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IS 4487 Assignment 11: Predicting Airbnb Prices with Regression**

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using Regularization
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## **Task 1. Load Your Transformed Airbnb Dataset**

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### 🔧 Write code for the following:

1.1. Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)

1.2. Use `pandas` to load and preview the dataset - call `.info()` and `.head()` and anything else you want



In [ ]:
# 1.1 Import your CSV file called cleaned_airbnb_data_7.csv
df = pd.read_csv('/content/cleaned_airbnb_data_7.csv')

# 1.2 Use pandas to load and preview the dataset - call .info() and .head()
print("=== Dataset Information ===")
df.info()

print("\n=== First 5 Rows ===")
display(df.head())

=== Dataset Information ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 97 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            500 non-null    int64  
 1   listing_url                                   500 non-null    object 
 2   scrape_id                                     500 non-null    int64  
 3   last_scraped                                  500 non-null    object 
 4   source                                        35 non-null     object 
 5   name                                          500 non-null    object 
 6   description                                   497 non-null    object 
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   500 non-null    object 
 9   host_id                              

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,reviews_per_month,price_log,minimum_nights_norm,price_std,review_volume,price_per_accommodates,is_long_term_stay,room_type_Entire home/apt,room_type_Private room,room_type_Shared room
0,2992450,https://www.airbnb.com/rooms/2992450,20260914164327,2026-09-16,NaN,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,...,0.06,4.200655,0.074176,-0.190344,Medium,16.4325,0,1,0,0
1,3820211,https://www.airbnb.com/rooms/3820211,20260914164327,2026-09-16,NaN,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,19648678,...,2.17,5.147494,0.000000,-0.102020,High,57.0000,0,1,0,0
2,6623339,https://www.airbnb.com/rooms/6623339,20260914164327,2026-09-16,NaN,Center Sq. Loft in Converted Precinct w/ Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,19648678,...,2.50,5.493061,0.000000,-0.042450,High,121.0000,0,1,0,0
3,9501054,https://www.airbnb.com/rooms/9501054,20260914164327,2026-09-16,NaN,Spacious suite with full bath by Capital Center,Great location within walking distance to the ...,NaN,https://a0.muscache.com/pictures/45153167-d704...,29288920,...,3.70,4.382027,0.000000,-0.179210,High,39.5000,0,0,1,0
4,10768745,https://www.airbnb.com/rooms/10768745,20260914164327,2026-09-16,NaN,Alb hospital area studio bath wifi. (Red),Spacious warm studio in 1840 house close to ho...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,5691268,...,7.28,3.931826,0.000000,-0.203542,High,50.0000,0,0,1,0


### ✍️ Please answer the following:

1.3. What business information does the dataset include?

1.4. How many rows and columns are present?


### ✍️ Your Response:
1.3. The dataset includes Airbnb prices, room types, locations, host information, availability, and customer reviews.

1.4. 500 rows and 97 columns.

## **Task 2. Drop Columns Not Useful for Modeling**

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### 🔧 Write code to do the following:

2.1. Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they're still in your dataset



In [ ]:
# 2.1. Drop columns like post_id, title, descr, details, and address if they're still in your dataset
# Map possible variants of these identifier/text columns present in this schema
columns_to_drop = [
    'id', 'scrape_id', 'listing_url', 'picture_url', 'host_id', 'host_url',
    'host_profile_id', 'host_profile_url', 'host_name', 'name', 'description',
    'neighborhood_overview', 'host_about', 'host_location', 'source',
    'last_scraped', 'first_review', 'last_review', 'calendar_last_scraped'
]

# Keep only those columns that actually exist in the DataFrame
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=existing_cols_to_drop)

print(f"Dropped {len(existing_cols_to_drop)} columns: {existing_cols_to_drop}")
print(f"Remaining columns: {df_cleaned.shape[1]}")

Dropped 19 columns: ['id', 'scrape_id', 'listing_url', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'name', 'description', 'neighborhood_overview', 'host_about', 'host_location', 'source', 'last_scraped', 'first_review', 'last_review', 'calendar_last_scraped']
Remaining columns: 78


### ✍️ Please answer the following:
2.2. What columns did you drop, and why?

2.3. What risks might occur if you included them in your model?


### ✍️ Your Response:
2.2. I dropped 19 columns, including IDs, URLs, names, descriptions, and dates. These columns are not very useful for the model.

2.3. These columns may add unnecessary information and cause overfitting. This can make the model less accurate.

## **Task 3. Explore Relationships Between Numeric Features**

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### 🔧 Write code to do the following:

3.1. Generate a correlation matrix




In [ ]:
# 3.1. Generate a correlation matrix
# Select only the numeric columns from the cleaned DataFrame
numeric_df = df_cleaned.select_dtypes(include=[np.number])

# Calculate the correlation matrix
correlation_matrix = numeric_df.corr()

# Preview correlations with 'price' specifically
if 'price' in correlation_matrix.columns:
    price_correlations = correlation_matrix['price'].sort_values(ascending=False)
    print("=== Correlations with Price ===")
    print(price_correlations)
else:
    print("'price' column not found in numeric features.")

# Display a snippet of the overall correlation matrix
print("\n=== Correlation Matrix Snippet (First 10 numeric features) ===")
display(correlation_matrix.iloc[:10, :10])

=== Correlations with Price ===
price_quote_price_per_night     1.000000
price                           1.000000
price_std                       1.000000
price_per_accommodates          0.976542
price_quote_total_price         0.932722
                                  ...   
host_verifications                   NaN
neighbourhood                        NaN
neighbourhood_group_cleansed         NaN
bathrooms                            NaN
instant_bookable                     NaN
Name: price, Length: 64, dtype: float64

=== Correlation Matrix Snippet (First 10 numeric features) ===


,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_response_time,host_response_rate,host_acceptance_rate,host_thumbnail_url,host_neighbourhood
host_since,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hosts_time_as_user_years,NaN,1.000000,0.116433,0.564641,0.078732,NaN,NaN,NaN,NaN,NaN
hosts_time_as_user_months,NaN,0.116433,1.000000,0.290172,0.249714,NaN,NaN,NaN,NaN,NaN
hosts_time_as_host_years,NaN,0.564641,0.290172,1.000000,-0.125667,NaN,NaN,NaN,NaN,NaN
hosts_time_as_host_months,NaN,0.078732,0.249714,-0.125667,1.000000,NaN,NaN,NaN,NaN,NaN
host_response_time,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_response_rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_acceptance_rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_thumbnail_url,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ✍️ Please answer the following questions:

3.2. Which variables had the strongest positive or negative correlation with price?

3.3. Which variables might be useful predictors of price?

### ✍️ Your Response:
3.3. Price, price_std, and price_quote_price_per_night have the strongest positive correlation

3.4. Room type, bedrooms, location, and accommodates may help predict price.

## **Task 4. Define Features and Target Variable**

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### 🔧 Write code to do the following:

4.1. Set `price` as your target variable

4.2. Remove `price` from your predictors



In [ ]:
# 4.1. Set price as your target variable
# We will handle missing values in 'price' first by dropping rows where it is NaN
df_ml = df_cleaned.dropna(subset=['price'])
y = df_ml['price']

# 4.2. Remove price from your predictors
# Additionally, we should remove variables that directly scale or contain price information
# to prevent data leakage (e.g., 'price_std', 'price_quote_price_per_night', 'price_per_accommodates', 'price_quote_total_price', 'price_log')
leaky_cols = ['price', 'price_std', 'price_quote_price_per_night', 'price_per_accommodates', 'price_quote_total_price', 'price_log']
existing_leaky_cols = [col for col in leaky_cols if col in df_ml.columns]

# Keep only numeric predictors, and fill any remaining NaNs with column medians
X = df_ml.drop(columns=existing_leaky_cols).select_dtypes(include=[np.number])
X = X.fillna(X.median())
# Drop columns that are completely empty/NaN after filtering
X = X.dropna(axis=1, how='all')

print(f"Target (y) shape: {y.shape}")
print(f"Predictors (X) shape: {X.shape}")

Target (y) shape: (467,)
Predictors (X) shape: (467, 46)


### ✍️ Please answer the following questions:

4.3. What features are you using as predictors?

4.4. Why is this a regression problem and not a classification problem?

### ✍️ Your Response:
4.3. I use 46 numeric features, including accommodates, bedrooms, availability, and number of reviews.

4.4. This is a regression problem because we are predicting price, which is a number, not a category.

## **Task 5. Split Data into Training and Testing Sets**


**Business framing:**

Splitting your data lets you train a model and test how well it performs on new, unseen data.

### 🔧 Write code to do the following:

5.1.  Use `train_test_split()` to split into 80% training, 20% testing

***CAVEAT: Ideally we should split our data before doing any transformations that are data dependent. However in this case, we will make an exception as our data was already processed/transformed, or we owuld have to start from scratch. Just remember to only perform pre-processing/cleaning tasks do not peak into the test set.***





In [ ]:
# 5.1. Use train_test_split() to split into 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

Training features shape: (373, 46)
Testing features shape: (94, 46)
Training target shape: (373,)
Testing target shape: (94,)


## **Task 6. Fit a Linear Regression Model**

**Business framing:**

Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### 🔧 Write code to do the following:

6.1. Fit a linear regression model to your training data

6.2. Use it to predict prices for the train and test set



In [ ]:
# 6.1. Fit a linear regression model to your training data
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 6.2. Use it to predict prices for the train and test set
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

print("Predictions completed. First 5 predictions on test set:")
print(y_test_pred[:5])

Predictions completed. First 5 predictions on test set:
[849.82808893 386.17670923 430.08734667 460.84539849 743.82279568]


## **Task 7. Evaluate Model Performance**

**Business framing:**

A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### 🔧 Write code to do the following:

7.1. Print MSE and R² score for your model


In [ ]:
# 7.1. Print MSE and R² score for your model
# Training Set Evaluation
mse_train = mean_squared_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

# Testing Set Evaluation
mse_test = mean_squared_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("=== Training Set Evaluation ===")
print(f"Mean Squared Error (MSE): {mse_train:.4f}")
print(f"R-squared (R²): {r2_train:.4f}")

print("\n=== Testing Set Evaluation ===")
print(f"Mean Squared Error (MSE): {mse_test:.4f}")
print(f"R-squared (R²): {r2_test:.4f}")

=== Training Set Evaluation ===
Mean Squared Error (MSE): 921906.7923
R-squared (R²): 0.3525

=== Testing Set Evaluation ===
Mean Squared Error (MSE): 1200300.3352
R-squared (R²): 0.1473


### ✍️ Please answer the following:

7.2. What is your R² score? How well does your model explain price variation?

7.3. Is your MSE large or small? What could you do to improve/lower it?

### ✍️ Your Response:

7.2. The test R² score is 0.1473. The model only explains about 14.7% of the price changes. It does not predict prices very well.

7.3. The test MSE is 1,200,300.34, which is large. We can use more useful features or try another model to lower it.

## **Task 8. Interpret Model Coefficients**

**Business framing:**

The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### 🔧 Write code to do the following:

8.1. Create a table showing feature names and regression coefficients

8.2. Sort the table so that the most impactful features (absolute magnitude-wise) are at the top


In [ ]:
# 8.1. Create a table showing feature names and regression coefficients
coefficients_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
})

# 8.2. Sort the table so that the most impactful features (absolute magnitude-wise) are at the top
coefficients_df['Abs_Coefficient'] = coefficients_df['Coefficient'].abs()
coefficients_df_sorted = coefficients_df.sort_values(by='Abs_Coefficient', ascending=False).drop(columns=['Abs_Coefficient'])

print("=== Top Most Impactful Features (Sorted by Absolute Coefficient Value) ===")
display(coefficients_df_sorted.head(20))

=== Top Most Impactful Features (Sorted by Absolute Coefficient Value) ===


,Feature,Coefficient
6,longitude,5167.701828
5,latitude,4044.069110
45,room_type_Shared room,1450.309988
44,room_type_Private room,-889.013073
43,room_type_Entire home/apt,-561.296915
39,calculated_host_listings_count_shared_rooms,-397.853729
29,review_scores_rating,393.086624
35,review_scores_value,-215.860144
34,review_scores_location,196.676968
38,calculated_host_listings_count_private_rooms,145.779360


### ✍️ Please answer the following:

8.3. Which features increased price the most?

8.4. Were any surprisingly negative?

8.5. What business insight could you draw from this?

### ✍️ Your Response:

8.3. Longitude, latitude, and shared room have the largest positive coefficients.

8.4. Yes. Bedrooms and accommodates have negative coefficients, which is surprising.

8.5. Location and room type are important for Airbnb prices. Hosts should consider these when setting prices.


## **Task 9. Try to Improve the Linear Regression Model**

**Business framing:**

The first version of your model included all available features — but not all features are equally useful. You can use automated feature engineering first and then regularization to either removing some features or shrink some coefficients, which can often improve test set performance and interpretation.

### 🔧 Write code to do the following:

9.1. Use feature engineering to create several more features. You can use automated or manual engineering.

9.2 Build and train a new regression model. Evaluate it on the train and test set.

9.3. Attempt to do some regularization. Use LassoCV or RidgeCV to find an optimal model by selecting the best value of the `alpha` hyperparameter.  

9.4. Print the optimal value of alpha.
Display the top coefficients for this best model.

9.5. Use this best regularized model to make predictions on the test set. Then evaluate this model on train and test set.



In [ ]:
# Import regularization classes from sklearn
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler

# 9.1. Use feature engineering to create several more features.
X_engineered = X.copy()

# Create interaction and ratio features safely
if 'accommodates' in X_engineered.columns and 'bedrooms' in X_engineered.columns:
    X_engineered['accommodates_per_bedroom'] = X_engineered['accommodates'] / (X_engineered['bedrooms'] + 1)
    X_engineered['accommodates_bedrooms_interaction'] = X_engineered['accommodates'] * X_engineered['bedrooms']

if 'number_of_reviews' in X_engineered.columns and 'review_scores_rating' in X_engineered.columns:
    X_engineered['review_volume_rating_interaction'] = X_engineered['number_of_reviews'] * X_engineered['review_scores_rating']

if 'availability_365' in X_engineered.columns:
    X_engineered['high_availability'] = (X_engineered['availability_365'] > 180).astype(int)

# Standardize features for regularized models
scaler = StandardScaler()
X_eng_scaled = pd.DataFrame(scaler.fit_transform(X_engineered), columns=X_engineered.columns)

# Re-split with the newly engineered features
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_eng_scaled, y, test_size=0.2, random_state=42
)

# 9.2 Build and train a new regression model. Evaluate it on the train and test set.
lr_eng_model = LinearRegression()
lr_eng_model.fit(X_train_eng, y_train_eng)

y_train_eng_pred = lr_eng_model.predict(X_train_eng)
y_test_eng_pred = lr_eng_model.predict(X_test_eng)

print("=== Baseline Regression with Engineered Features ===")
print(f"Train MSE: {mean_squared_error(y_train_eng, y_train_eng_pred):.4f} | R²: {r2_score(y_train_eng, y_train_eng_pred):.4f}")
print(f"Test MSE: {mean_squared_error(y_test_eng, y_test_eng_pred):.4f} | R²: {r2_score(y_test_eng, y_test_eng_pred):.4f}")

# 9.3. Attempt to do some regularization using RidgeCV
alphas = np.logspace(-3, 3, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_eng, y_train_eng)

# 9.4. Print the optimal value of alpha and display the top coefficients
optimal_alpha = ridge_cv.alpha_
print(f"\n=== Regularized Model (Ridge) ===")
print(f"Optimal Alpha: {optimal_alpha:.4f}")

# Table of top coefficients
ridge_coefs = pd.DataFrame({
    'Feature': X_engineered.columns,
    'Coefficient': ridge_cv.coef_
})
ridge_coefs['Abs_Coefficient'] = ridge_coefs['Coefficient'].abs()
ridge_coefs_sorted = ridge_coefs.sort_values(by='Abs_Coefficient', ascending=False).drop(columns=['Abs_Coefficient'])
print("\nTop 10 Coefficients for Regularized Model:")
display(ridge_coefs_sorted.head(10))

# 9.5. Use this best regularized model to make predictions and evaluate
y_train_ridge_pred = ridge_cv.predict(X_train_eng)
y_test_ridge_pred = ridge_cv.predict(X_test_eng)

print("\n=== Regularized Model Evaluation ===")
print(f"Train MSE: {mean_squared_error(y_train_eng, y_train_ridge_pred):.4f} | R²: {r2_score(y_train_eng, y_train_ridge_pred):.4f}")
print(f"Test MSE: {mean_squared_error(y_test_eng, y_test_ridge_pred):.4f} | R²: {r2_score(y_test_eng, y_test_ridge_pred):.4f}")

=== Baseline Regression with Engineered Features ===
Train MSE: 920831.0449 | R²: 0.3533
Test MSE: 1201117.2161 | R²: 0.1467

=== Regularized Model (Ridge) ===
Optimal Alpha: 53.3670

Top 10 Coefficients for Regularized Model:


,Feature,Coefficient
28,estimated_revenue_l365d,524.496722
27,estimated_occupancy_l365d,-272.200437
4,host_listings_count,244.331033
3,hosts_time_as_host_months,-151.959212
1,hosts_time_as_user_months,103.734051
9,beds,-92.590298
34,review_scores_location,90.942886
6,longitude,87.255606
35,review_scores_value,-80.325054
0,hosts_time_as_user_years,-79.629149



=== Regularized Model Evaluation ===
Train MSE: 964498.2206 | R²: 0.3226
Test MSE: 1169946.2518 | R²: 0.1689


### ✍️ Please answer the following:

9.6. Compared to the model with the full set of engineerined features, does the regularized model perform better or worse?

9.7. Does the linear regression model used on this dataset suffer from overfitting or underfitting or neither? Which of these two models  does better at dealing with a) overfitting and b) underfitting?


### ✍️ Your Response:

9.6. The regularized model performs better. It has a lower test MSE and a higher R² score

9.7. The linear regression model shows some overfitting. The regularized model is better for overfitting, while the original model is better for avoiding underfitting.



## **10. Reflect and Recommend**

**Business framing:**

Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### ✍️ Please answer the following questions:

10.1. What business question did your model help answer?

10.2. What would you recommend to Airbnb or its hosts? Any insights on useful features?

10.3. What could you do next to improve this model or make it more useful?

10.4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧

10.1. The model helps predict Airbnb prices based on different features.

10.2. Hosts should consider location, room type, and reviews when setting prices.

10.3. I can use more data and try different models to improve the predictions.

10.4. This lab helps me learn how to use linear regression and Ridge regression to predict prices. I also learned how to compare models using MSE and R².

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_Gong Linlin.ipynb"

[NbConvertApp] Converting notebook assignment_11_Gong Linlin.ipynb to html
[NbConvertApp] Writing 374828 bytes to assignment_11_Gong Linlin.html
